<a href="https://colab.research.google.com/github/AndresCMontejo/Veterinaria_DOGtor_Agente_IA/blob/main/Dokky_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**¿COMO SE CONSTRUYÓ EL AGENTE?**

##Pasos
1. Conectar Drive
2. Encontrar los Docx
3. Extraer el texto
4. Añadir metadatos
5. Separar el system prompt
6. Dividir el texto en chunks
7. Generar embeddings
8. Guardarlos en Chroma
9. Hacer búsquedas de prueba
10. Conectar Gemini para redactar respuestas
11. Crear la interfaz

**1. Conectar Drive**

In [5]:
#Conectando con google drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**2. Encontrar los GDOC**

In [6]:
#Definiendo la ruta de los documentos
from pathlib import Path
DOCUMENTS_DIR = Path(
    "/content/drive/MyDrive/documentos_agente_de_ia/documentos"
)

if not DOCUMENTS_DIR.exists():
    raise FileNotFoundError(
        f"No se encontró la carpeta: {DOCUMENTS_DIR}"
    )

print("Carpeta encontrada:", DOCUMENTS_DIR)

Carpeta encontrada: /content/drive/MyDrive/documentos_agente_de_ia/documentos


In [13]:
#COMPROBANDO SI HAY ARCHIVOS EXISTENTES EN DOCUMENTOS
document_files = list(DOCUMENTS_DIR.glob("*.docx"))

print(f"Documentos encontrados: {len(document_files)}")

for file in document_files:
    print("-", file.name)

Documentos encontrados: 6
- servicios_veterinaria_dogtor.docx
- guia_convenios.docx
- politica_cancelaciones_reprogramacion.docx
- politica_privacidad_datos_paciente.docx
- instrucciones_pre_postconsulta.docx
- citas_agendamientos.docx


In [11]:
#Preparación de bibliotecas
!pip install -qU \
    python-docx \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-chroma \
    chromadb \
    google-genai

In [12]:
#Comprobando la llave API Key de gemini
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI")
if not GEMINI_API_KEY:
    raise ValueError(
        "No se encontró GEMINI_API_KEY en los secretos de Colab."
    )

print("Clave de Gemini cargada correctamente.")

Clave de Gemini cargada correctamente.


In [14]:
#COMPROBANDO EL FORMATO DE LOS ARCHIVOS
from pathlib import Path
BASE_DIR = Path(
    "/content/drive/MyDrive/documentos_agente_de_ia"
)
DOCUMENTS_DIR = BASE_DIR / "documentos"
PROMPT_DIR = BASE_DIR / "prompt"
print("Archivos de conocimiento:")
for archivo in DOCUMENTS_DIR.iterdir():
    print(
        f"- {archivo.name} | "
        f"Extensión: {archivo.suffix} | "
        f"Tamaño: {archivo.stat().st_size} bytes"
    )

print("\nArchivos de configuración:")
for archivo in PROMPT_DIR.iterdir():
    print(
        f"- {archivo.name} | "
        f"Extensión: {archivo.suffix} | "
        f"Tamaño: {archivo.stat().st_size} bytes"
    )

Archivos de conocimiento:
- DOCUMENTOS GDOC | Extensión:  | Tamaño: 4096 bytes
- servicios_veterinaria_dogtor.docx | Extensión: .docx | Tamaño: 9157 bytes
- guia_convenios.docx | Extensión: .docx | Tamaño: 12184 bytes
- politica_cancelaciones_reprogramacion.docx | Extensión: .docx | Tamaño: 321292 bytes
- politica_privacidad_datos_paciente.docx | Extensión: .docx | Tamaño: 11043 bytes
- instrucciones_pre_postconsulta.docx | Extensión: .docx | Tamaño: 12948 bytes
- citas_agendamientos.docx | Extensión: .docx | Tamaño: 11079 bytes

Archivos de configuración:
- Prompt GDOC | Extensión:  | Tamaño: 4096 bytes
- system_prompt.docx | Extensión: .docx | Tamaño: 9132 bytes


In [15]:
#COMPROBANDO LAS DEPENDENCIAS INSTALADAS
import docx
import chromadb
import langchain
import google.genai

print("python-docx:", docx.__version__)
print("chromadb:", chromadb.__version__)
print("langchain:", langchain.__version__)
print("Las bibliotecas principales están disponibles.")

python-docx: 1.2.0
chromadb: 1.5.9
langchain: 1.3.14
Las bibliotecas principales están disponibles.


**3. Extraer el texto**

In [25]:
#El siguiente codigo se utiliza para extraer el contenido de los archivos .docx, y las tablas, ya que cualquier error, el agente puede no funcionar como se espera.
from pathlib import Path
from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table

def extraer_texto_docx(ruta_archivo: Path) -> str:
    """
    Extrae párrafos y tablas de un archivo DOCX
    conservando el orden original del documento.
    """
    documento = Document(ruta_archivo)
    partes = []

    for elemento in documento.iter_inner_content():

        # Extraer párrafos
        if isinstance(elemento, Paragraph):
            texto = elemento.text.strip()

            if texto:
                partes.append(texto)

        # Extraer tablas
        elif isinstance(elemento, Table):
            partes.append("[INICIO DE TABLA]")

            for fila in elemento.rows:
                celdas = []

                for celda in fila.cells:
                    texto_celda = " ".join(
                        parrafo.text.strip()
                        for parrafo in celda.paragraphs
                        if parrafo.text.strip()
                    )

                    celdas.append(texto_celda)

                if any(celdas):
                    partes.append(" | ".join(celdas))

            partes.append("[FIN DE TABLA]")

    return "\n".join(partes)

**4. Añadir metadatos**

In [26]:
#CREANDO METADATOS DE LOS DOCUMENTOS
METADATOS_DOCUMENTOS = {
    "servicios_veterinaria_dogtor.docx": {
        "Categoria": "Servicios",
        "Departamento responsable": "Dirección Médica",
        "Nivel de acceso": "Público",
        "Estado": "Vigente",
        "Versión": "1.0"
    },

    "politica_privacidad_datos_paciente.docx": {
        "Categoria": "Legal y Compliance",
        "Departamento responsable": "Departamento Jurídico",
        "Tipo de documento": "Política Corporativa",
        "Nivel de acceso": "Todos los colaboradores",
        "Estado": "Vigente",
        "Version": "1.0"
    },

    "politica_cancelaciones_reprogramacion.docx": {
        "Categoria": "Atención al Cliente",
        "Departamento responsable": "Recepción y Atención al Cliente",
        "Tipo de documento": "Política Corporativa",
        "Nivel de acceso": "Todos los colaboradores",
        "Estado": "Vigente",
        "Version": "1.0"
    },

    "instrucciones_pre_postconsulta.docx": {
        "Categoria": "Atención médica",
        "Departamento responsable": "Dirección médica",
        "Tipo de documento": "Política Corporativa",
        "Nivel de acceso": "Manual clínico",
        "Estado": "Vigente",
        "Version":"1.0"
    },

    "guia_convenios.docx": {
        "Categoria": "Convenios y Alianzas",
        "Departamento responsable": "Relaciones institucionales",
        "Tipo de documento": "Guia corporativa",
        "Nivel de acceso": "Todo los colaboradores",
        "Estado": "Vigente",
        "Version": "1.0"
    },

    "citas_agendamientos.docx": {
        "Categoria": "Atención al cliente",
        "Departamento responsable": "Recepción y Atención al Cliente",
        "Tipo de documento": "Preguntas Frecuentes",
        "Nivel de acceso": "Todos los colaboradores",
        "Estado": "Vigente",
        "Versión":"1.0"
    }
}

In [27]:
#CARGANDO LOS DOCUMENTOS
documentos_cargados = []

archivos_docx = sorted(DOCUMENTS_DIR.glob("*.docx"))

for ruta_archivo in archivos_docx:
    texto = extraer_texto_docx(ruta_archivo)

    metadatos = METADATOS_DOCUMENTOS.get(
        ruta_archivo.name,
        {
            "Categoria": "Sin clasificar",
            "Departamento responsable": "Sin especificar",
            "Tipo de documento": "Documento",
            "Nivel de acceso": "Público",
            "Estado": "Vigente",
            "Version": "1.0"
        }
    )

    metadatos = {
        **metadatos,
        "nombre_archivo": ruta_archivo.name,
        "ruta_origen": str(ruta_archivo),
        "formato": "docx"
    }

    documentos_cargados.append(
        {
            "texto": texto,
            "metadata": metadatos
        }
    )

print(f"Documentos cargados: {len(documentos_cargados)}")

Documentos cargados: 6


In [28]:
#PRUEBA, PARA LOS DOCUMENTOS QUE TENGAN UNICAMENTE TABLAS
for documento in documentos_cargados:
    texto = documento["texto"]
    archivo = documento["metadata"]["nombre_archivo"]

    cantidad_tablas = texto.count("[INICIO DE TABLA]")

    print(
        f"{archivo}: "
        f"{cantidad_tablas} tabla(s) detectada(s)"
    )

citas_agendamientos.docx: 0 tabla(s) detectada(s)
guia_convenios.docx: 1 tabla(s) detectada(s)
instrucciones_pre_postconsulta.docx: 2 tabla(s) detectada(s)
politica_cancelaciones_reprogramacion.docx: 1 tabla(s) detectada(s)
politica_privacidad_datos_paciente.docx: 0 tabla(s) detectada(s)
servicios_veterinaria_dogtor.docx: 0 tabla(s) detectada(s)


In [29]:
#MOSTRANDO LOS DOCUMENTOS QUE TENGAN UNICAMENTE TABLAS
for documento in documentos_cargados:
    texto = documento["texto"]
    archivo = documento["metadata"]["nombre_archivo"]

    if "[INICIO DE TABLA]" in texto:
        posicion = texto.find("[INICIO DE TABLA]")

        print("=" * 80)
        print("Archivo:", archivo)
        print("\nFragmento con tabla:\n")

        inicio = max(0, posicion - 200)
        fin = min(len(texto), posicion + 1200)

        print(texto[inicio:fin])
        print()

Archivo: guia_convenios.docx

Fragmento con tabla:

ROS DE GASTOS MÉDICOS)
Aceptamos y facturamos directamente a las siguientes aseguradoras. Si tu póliza está activa, no pagas en caja (solo el deducible o coaseguro que marque tu contrato, si aplica).
[INICIO DE TABLA]
Aseguradora | Tipo de cobertura | Beneficio DOGtor | Proceso de cobro
GNP Seguros (Póliza Mascotas) | Accidentes, cirugías, hospitalización y estudios de diagnóstico. | Convenio preferente: Sin tope de gastos notariales para la reclamación. | Cobramos directamente a GNP. Tú solo firmas la factura y pagas el deducible establecido.
Mapfre México (Protección Animal) | Consultas de especialidad, medicamentos recetados y urgencias. | 10% de descuento en el deducible si pagas con tarjeta DOGtor (afiliación gratuita). | Emitimos factura CFDI a nombre de Mapfre y te damos el comprobante para tu ajuste.
AXA Seguros (Salud para Mascotas) | Cobertura integral (incluye consultas generales y vacunación anual). | Cobertura ampliada: I

In [30]:
#Comprobando cada documento
for numero, documento in enumerate(documentos_cargados, start=1):
    texto = documento["texto"]
    metadata = documento["metadata"]

    print("=" * 80)
    print(f"DOCUMENTO {numero}")
    print("Archivo:", metadata["nombre_archivo"])
    print("Categoría:", metadata["Categoria"])
    print("Departamento:", metadata["Departamento responsable"])
    print("Caracteres extraídos:", len(texto))
    print("\nVista previa:")
    print(texto[:7000])
    print()

DOCUMENTO 1
Archivo: citas_agendamientos.docx
Categoría: Atención al cliente
Departamento: Recepción y Atención al Cliente
Caracteres extraídos: 6438

Vista previa:
PREGUNTAS FRECUENTES (FAQ) – CITAS Y AGENDAMIENTOS
VERSIÓN: 1.0
FECHA DE VIGENCIA: 8 de Octubre de 2026
EMPRESA: Veterinaria DOGtor S.A. de C.V.
DIRECCIÓN SEDE CENTRAL: Av. Paseo de la Reforma 1234, Colonia Juárez, Alcaldía Cuauhtémoc, CDMX.
Categoría: Atención al cliente
Departamento responsable: Recepción y Atención al Cliente
Tipo de documento: Preguntas Frecuentes
Nivel de acceso: Todos los colaboradores
Estado: Vigente
Palabras clave:
Citas, horarios, costos, agendar, urgencias
INTRODUCCIÓN
En Veterinaria DOGtor sabemos que la salud de tu peludo es prioridad. Por eso, hemos diseñado un sistema de agendamiento ágil, transparente y adaptado a tu ritmo de vida en la Ciudad de México. A continuación, resolvemos las dudas más comunes para que agendar, modificar o prepararte para tu cita sea pan comido (o mejor dicho, croque

**5. Separar el system prompt**

In [31]:
#CARGANDO EL SYSTEM PROMPT
archivos_prompt = list(PROMPT_DIR.glob("*.docx"))

if len(archivos_prompt) != 1:
    raise ValueError(
        f"Se esperaba un único archivo DOCX en la carpeta prompt, "
        f"pero se encontraron {len(archivos_prompt)}."
    )

ruta_system_prompt = archivos_prompt[0]
SYSTEM_PROMPT = extraer_texto_docx(ruta_system_prompt)

print("System prompt cargado correctamente.")
print("Archivo:", ruta_system_prompt.name)
print("Caracteres:", len(SYSTEM_PROMPT))
print("\nVista previa:")
print(SYSTEM_PROMPT[:700])

System prompt cargado correctamente.
Archivo: system_prompt.docx
Caracteres: 3952

Vista previa:
PROMPT DE SISTEMA PARA EL AGENTE VETERINARIO DOGtor
Versión: 1.0 | Fecha: 26/07/2026
1. Rol y Personalidad:
Eres "Dokky", el asistente virtual oficial de Veterinaria DOGtor S.A. de C.V., ubicada en Av. Paseo de la Reforma 1234, CDMX. Tu misión es brindar información clara, cálida y precisa a los tutores (dueños) de mascotas. Habla con empatía, usa un tono amigable pero profesional, y ocasionalmente puedes usar emojis de perros o gatos 🐶🐱 para dar calidez, pero sin perder la formalidad médica cuando sea necesario.
2. Reglas de Oro (Comportamiento General):
NUNCA des diagnósticos médicos, pronósticos ni recetas. Eres un asistente informativo, no un veterinario. Si el usuario describe síntomas 
